### How to Resume in a New Session

If your runtime disconnects, follow these steps to pick up where you left off:
1. Run **Cell 1** to mount Drive.
2. Run **Cell 2** to install requirements.
3. Run **Cell 5** to copy the CSVs to local SSD (Jan + Feb + Mar, required every new session).
4. In **Cell 6**, change `WEEKS_TO_RUN` to the remaining weeks (e.g. `[5, 6, 7, 8]`) and run.

The pipeline saves `pipeline_state.json` to Drive after each week, so completed weeks are
automatically skipped on resume.

**Starting completely fresh?** Delete `MyDrive/recosys_weekly/pipeline_state.json` and run
Cell 6 with `WEEKS_TO_RUN = list(range(1, 9))` (or uncomment `--reset` in the Popen args).

# Chapter 12 — Weekly Iterative Fine-Tuning (Rolling Eval + Experience Replay)

**Goal:** Fine-tune GRU4Rec V9 on incremental Feb + Mar 2020 data in 8 weekly steps,
using production-grade evaluation methodology.

**Evaluation strategy — rolling val set:**
Each week W fine-tunes on events [start_W, end_W), then evaluates on the *next week's*
sessions [end_W, end_{W+1}). This mirrors production: the model is judged on whether it
predicts future behaviour, not whether it remembers January. Week 8 falls back to the
original `test_sessions.parquet`.

**Catastrophic forgetting mitigation — experience replay:**
Each week's fine-tuning batch mixes 30% random sessions from the original
`train_sessions.parquet` to keep the model anchored to the full 2.9M session distribution.

**Drift baseline:**
- Weeks 1–4 (Feb): JSD vs Jan 2020 distribution
- Weeks 5–8 (Mar): JSD vs Feb 2020 distribution (COVID spike expected around week 6)

Each weekly run:
1. Computes item-popularity drift (JSD) vs. monthly baseline
2. Fine-tunes 5 epochs from current checkpoint at `lr=1e-4` + experience replay
3. Evaluates on next week's sessions (rolling eval)
4. Promotes checkpoint only if rolling NDCG@20 improves ≥ 0.0005

**Runtime:** ~15–18h total on T4 GPU (all 8 weeks). State is saved to Drive after each week.

## Prerequisites
- Google Drive mounted with REES46 CSVs at `MyDrive/rees46/` (Jan, Feb, **and Mar**)
- Session parquets at `MyDrive/recosys_sessions/` (train + test)
- Colab runtime set to **T4 GPU** (Runtime → Change runtime type → T4 GPU)
- **IMPORTANT:** Delete `MyDrive/recosys_weekly/pipeline_state.json` if you ran the old
  frozen-val-set version — the new rolling eval strategy is incompatible with the old state.

In [ ]:
# ── Cell 1: Mount Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 2: Clone / update repo + install deps ───────────────────────────────
import os

if not os.path.exists('/content/RecoSys'):
    !git clone -b dev https://github.com/sbnikhil/RecoSys.git /content/RecoSys
else:
    !git -C /content/RecoSys pull origin dev

%cd /content/RecoSys
!pip install torch torchvision pandas pyarrow numpy huggingface_hub faiss-cpu -q

Cloning into '/content/RecoSys'...
remote: Enumerating objects: 576, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 576 (delta 107), reused 179 (delta 64), pack-reused 338 (from 1)
Receiving objects: 100% (576/576), 1.50 MiB | 7.33 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/RecoSys
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 99.4 MB/s eta 0:00:00


In [ ]:
# ── Cell 3: Verify GPU ───────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — switch to T4 GPU runtime!')

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# ── Cell 4: Download model artifacts from HF Hub ────────────────────────────
from huggingface_hub import hf_hub_download
import shutil, os

os.makedirs('model', exist_ok=True)

for fname in ['model_inference.pt', 'vocabs.pkl']:
    local = f'model/{fname}'
    if os.path.exists(local):
        print(f'  Already exists: {fname}')
        continue
    tmp = hf_hub_download(
        repo_id='manojarulmurugan/recosys-models',
        filename=fname,
        repo_type='dataset',
    )
    shutil.copy(tmp, local)
    print(f'  Downloaded {fname}: {round(os.path.getsize(local)/1e6, 1)} MB')

model_inference.pt:   0%|          | 0.00/115M [00:00<?, ?B/s]

  Downloaded model_inference.pt: 115.4 MB


vocabs.pkl:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

  Downloaded vocabs.pkl: 21.8 MB


In [ ]:
# ── Cell 5: Copy CSVs to local SSD (faster I/O than FUSE mount) ─────────────
import shutil, os, time

src = '/content/drive/MyDrive/rees46'
dst = '/content/rees46_local'
os.makedirs(dst, exist_ok=True)

# Jan + Feb + Mar all needed: Jan for drift baseline, Feb+Mar for training and rolling eval
for f in ['2020-Jan.csv.gz', '2020-Feb.csv.gz', '2020-Mar.csv.gz']:
    d = f'{dst}/{f}'
    if os.path.exists(d):
        print(f'  Already local: {f}')
        continue
    s = f'{src}/{f}'
    if not os.path.exists(s):
        print(f'  NOT FOUND on Drive: {s}  ← upload this file first!')
        continue
    sz = os.path.getsize(s) / 1e6
    print(f'  Copying {f} ({sz:.0f} MB)...', end=' ', flush=True)
    t = time.time()
    shutil.copy2(s, d)
    print(f'done in {time.time()-t:.0f}s')

print('\nLocal files:', sorted(os.listdir(dst)))

  Copying 2020-Jan.csv.gz (2390 MB)... done in 52s
  Copying 2020-Feb.csv.gz (2347 MB)... done in 55s
  Copying 2020-Mar.csv.gz (2417 MB)... done in 72s

Local files: ['2020-Feb.csv.gz', '2020-Jan.csv.gz', '2020-Mar.csv.gz']


In [ ]:
# ── Cell 6: Run weekly pipeline — rolling eval + experience replay ────────────
#
# WEEKS_TO_RUN controls which weeks execute this session.
# Weeks 1-4 = February 2020.  Weeks 5-8 = March 2020.
#
# The script saves pipeline_state.json to Drive after each week, so it's safe
# to interrupt and resume (just change WEEKS_TO_RUN to the remaining weeks).
#
# First run ever? Leave WEEKS_TO_RUN = list(range(1, 9)) to run all 8 weeks.
# Resuming? Change to e.g. [5, 6, 7, 8] to continue from March.
#
# If you previously ran the OLD frozen-val-set version, you MUST delete
# MyDrive/recosys_weekly/pipeline_state.json before running — or add --reset below.

import subprocess, sys, time

WEEKS_TO_RUN = list(range(1, 9))   # ← change to [5,6,7,8] to resume from March

start = time.time()

proc = subprocess.Popen(
    [
        sys.executable,
        'scripts/retrain/run_weekly_pipeline.py',
        # ── Data sources ─────────────────────────────────────────
        '--baseline-csv',    '/content/rees46_local/2020-Jan.csv.gz',
        '--feb-csv',         '/content/rees46_local/2020-Feb.csv.gz',
        '--mar-csv',         '/content/rees46_local/2020-Mar.csv.gz',
        # Historical sessions for experience replay (30% mix)
        '--train-sessions',  '/content/drive/MyDrive/recosys_sessions/train_sessions.parquet',
        # Fallback eval for week 8 (no week 9 data)
        '--test-sessions',   '/content/drive/MyDrive/recosys_sessions/test_sessions.parquet',
        # ── Model ────────────────────────────────────────────────
        '--base-ckpt',       'model/model_inference.pt',
        '--vocabs-path',     'model/vocabs.pkl',
        '--ckpt-dir',        '/content/drive/MyDrive/recosys_weekly',
        # ── Pipeline control ──────────────────────────────────────
        '--weeks',           *[str(w) for w in WEEKS_TO_RUN],
        '--replay-ratio',    '0.3',
        '--finetune-epochs', '5',
        '--lr',              '1e-4',
        # Uncomment --reset if you need to wipe old frozen-val state:
        # '--reset',
    ],
    cwd='/content/RecoSys',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
elapsed = (time.time() - start) / 60
print(f'\nTotal time: {elapsed:.1f} min  |  Exit code: {proc.returncode}')
if proc.returncode != 0:
    print('Pipeline failed — scroll up for the error.')


RecoSys — Weekly Fine-Tuning Pipeline
  Device        : cuda
  Weeks         : [1, 2, 3, 4, 5, 6, 7, 8]
  Epochs/week   : 5
  LR            : 0.0001
  Replay ratio  : 30%
  Eval strategy : rolling (week W → eval on week W+1)
  MLflow not installed — skipping experiment logging

  ▶ Loading vocabs ...
     done in 0m 0s — 890,736 users / 222,864 items

  ▶ Loading 2020-Jan.csv.gz ...
     done in 3m 55s — 7,139,619 in-vocab events

  ▶ Loading 2020-Feb.csv.gz ...
     done in 3m 36s — 5,277,119 in-vocab events

  ▶ Loading 2020-Mar.csv.gz ...
     done in 3m 27s — 4,260,220 in-vocab events

  ▶ Loading historical sessions for replay buffer ...
     done in 0m 8s — 2,887,783 historical sessions available

  ▶ Loading test sessions (week-8 fallback) ...
     done in 0m 2s — 515,358 sessions

  Copied base checkpoint → /content/drive/MyDrive/recosys_weekly/current_best.pt

  Evaluating baseline model on rolling-eval week 1 (Feb 08–15) ...
  SessionEvalDataset: 139,463 sessions, prefix sha

In [ ]:
# ── Cell 8: Inspect pipeline state ──────────────────────────────────────────
import json, os

state_path = '/content/drive/MyDrive/recosys_weekly/pipeline_state.json'
if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    print('Pipeline state:')
    print(json.dumps(state, indent=2))
else:
    print('No pipeline_state.json yet — run Cell 6+7 first')

Pipeline state:
{
  "current_ndcg_20": 0.27138128876686096,
  "eval_strategy": "rolling",
  "history": [
    {
      "week": 1,
      "jsd": 0.1215168647731117,
      "action": "promoted",
      "eval_on": "week 2 (2020-02-08\u20132020-02-15)",
      "val_ndcg_20": 0.26674792170524597,
      "improvement": 0.007559478282928467,
      "epoch_losses": [
        7.359710221373996,
        7.158003445165066,
        7.053868646971316,
        6.988491281992678,
        6.9515952098695974
      ],
      "hr_20": 0.49115535616874695
    },
    {
      "week": 2,
      "jsd": 0.1462177516754052,
      "action": "skipped",
      "eval_on": "week 3 (2020-02-15\u20132020-02-22)",
      "val_ndcg_20": 0.25716719031333923,
      "improvement": -0.009580731391906738,
      "epoch_losses": [
        7.295527958347371,
        7.144226001592574,
        7.047324828186095,
        6.985499910358414,
        6.950765542424269
      ],
      "hr_20": 0.4790079891681671
    },
    {
      "week": 3,
    

In [ ]:
# ── Cell 9 (optional): Upload best checkpoint to HF Hub ────────────────────
import subprocess, sys

HF_TOKEN = ''  # ← paste your HF token here (Settings → Access Tokens on huggingface.co)

if HF_TOKEN:
    subprocess.run([
        sys.executable, 'scripts/deploy/upload_to_hf_hub.py',
        '--token',        HF_TOKEN,
        '--hf-username',  'manojarulmurugan',
        '--repo-name',    'recosys-models',
        '--model-dir',    '/content/drive/MyDrive/recosys_weekly',
    ], cwd='/content/RecoSys')
else:
    print('Set HF_TOKEN above to push the checkpoint to HF Hub')